# 04 - 数据库层与多表关联

> **何时使用**: 当你有多个相互关联的表，需要确保外键完整性时。
>
> **核心概念**: sqlseed 自动检测表依赖关系，按拓扑顺序填充，通过 SharedPool 跨表共享值。

## 适用场景

- 多表有 FOREIGN KEY 约束 → sqlseed 自动排序填充
- 两表共享同名列（如 `member_no`）→ SharedPool 隐式关联
- 列名不同但需要关联（如 `department_id` → `id`）→ ColumnAssociation 显式关联
- 大数据量写入性能优化 → Pragma 三级优化

## 你将学到

- 双适配器架构（sqlite-utils / raw sqlite3）
- Pragma 三级写入优化
- SharedPool 跨表值共享
- ColumnAssociation 显式关联
- BLOB / 大文本处理

详见 architecture.zh-CN.md §5

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| **→ 04** | **数据库层与多表关联** | **Database + Core** | **01** |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
from pathlib import Path

# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 关系解析 | `src/sqlseed/core/relation.py` | `RelationResolver` |
| 数据库接口 | `src/sqlseed/database/_protocol.py` | `DatabaseAdapter` |

> 对应架构图: [§5 数据库层架构](../docs/architecture.zh-CN.md#5-数据库层架构)

## 1. 先看效果 — 多表关联自动排序

数据库有外键约束？sqlseed 自动检测表依赖关系，按拓扑顺序填充 — **无需手动指定顺序**：

In [2]:
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# 一次填充 5 张表 — sqlseed 自动处理 FK 顺序
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name='organizations', count=3, clear_before=True),
        TableConfig(name='members', count=10, clear_before=True),
        TableConfig(name='projects', count=5, clear_before=True),
        TableConfig(name='tasks', count=20, clear_before=True),
        TableConfig(name='tags', count=5, clear_before=True),
    ]
)
config_path = Path('_fk_demo.yaml')
save_config(config, str(config_path))

results = fill_from_config(str(config_path))
print(f"{'表名':<15s}  {'行数':>6s}  {'耗时':>8s}  {'速度':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/20 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/5 [00:00<?, ?it/s]

表名                   行数        耗时          速度
---------------------------------------------
organizations         3    0.073s        41 rows/s
members              10    0.053s       190 rows/s
projects              5    0.047s       107 rows/s
tasks                20    0.043s       468 rows/s
tags                  5    0.043s       117 rows/s


注意输出顺序 — `organizations` 先于 `members`，`projects` 先于 `tasks`。sqlseed 通过拓扑排序自动确定了正确的填充顺序，确保外键引用有效。

下面详细拆解数据库层的每个机制。

## 2. 双适配器架构

sqlseed 支持两种数据库适配器：

| 适配器 | 依赖 | 默认 | 特点 |
|--------|------|:----:|------|
| `SQLiteUtilsAdapter` | sqlite-utils | ✅ | 功能丰富，API 简洁 |
| `RawSQLiteAdapter` | 无（标准库） | - | 零依赖，兼容性好 |

sqlseed 自动检测：如果安装了 `sqlite-utils`，使用 `SQLiteUtilsAdapter`；否则降级到 `RawSQLiteAdapter`。

In [3]:
from sqlseed.database._compat import HAS_SQLITE_UTILS

print(f"sqlite-utils available: {HAS_SQLITE_UTILS}")
print(f"Active adapter: {'SQLiteUtilsAdapter' if HAS_SQLITE_UTILS else 'RawSQLiteAdapter'}")

sqlite-utils available: True
Active adapter: SQLiteUtilsAdapter


## 3. Pragma 优化

sqlseed 在批量写入时自动优化 SQLite Pragma 设置，提升性能：

| 级别 | journal_mode | synchronous | locking_mode | 适用场景 |
|------|-------------|-------------|-------------|----------|
| light | WAL | NORMAL | - | 小数据量（<1K 行） |
| moderate | WAL | OFF | - | 中等数据量（1K-10K 行） |
| aggressive | MEMORY | OFF | EXCLUSIVE | 大数据量（>10K 行） |

通过 `optimize_pragma=True`（默认）启用，退出时自动恢复原始设置。

In [4]:

# Pragma 优化在批量写入时自动调整 journal_mode, synchronous 等参数
# 效果在大数据量下更明显, 小数据量受 UNIQUE 约束求解开销影响较大
result_no_opt = fill(str(db_path), table="tasks", count=3000, optimize_pragma=False, clear_before=True)
print(f"Without optimization: {result_no_opt.count} rows in {result_no_opt.elapsed:.3f}s ({result_no_opt.rows_per_second:.0f} rows/s)")  # noqa: E501

result = fill(str(db_path), table="tasks", count=3000, optimize_pragma=True, clear_before=True)
print(f"With Pragma optimization: {result.count} rows in {result.elapsed:.3f}s ({result.rows_per_second:.0f} rows/s)")

Generating tasks:   0%|          | 0/3000 [00:00<?, ?it/s]

Without optimization: 3000 rows in 0.433s (6927 rows/s)


Generating tasks:   0%|          | 0/3000 [00:00<?, ?it/s]

With Pragma optimization: 3000 rows in 0.628s (4775 rows/s)


## 4. FK 解析：显式 vs 隐式

### 显式 FK

在 CREATE TABLE 中声明的 FOREIGN KEY 约束：

```sql
CREATE TABLE tasks (
    project_id INTEGER NOT NULL,
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);
```

sqlseed 自动检测显式 FK，生成引用父表已有值的数据。

### 隐式 FK

没有声明 FOREIGN KEY，但列名与父表主键同名（如 `member_id` 在 `reviews` 表中）。sqlseed 通过名称匹配自动推断关联。

In [5]:
import sqlite3

conn = sqlite3.connect(str(db_path))

print("--- reviews 表 FK ---")
fk_list = conn.execute("PRAGMA foreign_key_list(reviews)").fetchall()
print(f"Explicit FKs: {fk_list}")
print("Note: member_id has no explicit FK, but sqlseed detects it via name matching")

conn.close()

--- reviews 表 FK ---
Explicit FKs: [(0, 0, 'tasks', 'task_id', 'task_id', 'NO ACTION', 'NO ACTION', 'NONE')]
Note: member_id has no explicit FK, but sqlseed detects it via name matching


## 5. SharedPool 跨表值共享

当多张表引用同一张父表时，`SharedPool` 确保引用一致性：

- `tasks.project_id` 引用 `projects.project_id` 池（显式 FK）
- `tasks.assignee_id` 引用 `members.member_id` 池（显式 FK）
- `reviews.task_id` 引用 `tasks.task_id` 池（显式 FK）
- `reviews.member_id` 引用 `members.member_id` 池（隐式 FK，名称匹配）

详见 architecture.md §5 SharedPool

In [6]:

with connect(str(db_path)) as orch:
    r1 = orch.fill_table("organizations", count=3)
    r2 = orch.fill_table("members", count=10)
    r3 = orch.fill_table("projects", count=5)
    r4 = orch.fill_table("tasks", count=20)
    r5 = orch.fill_table("reviews", count=10)

conn = sqlite3.connect(str(db_path))
project_ids_tasks = {r[0] for r in conn.execute("SELECT DISTINCT project_id FROM tasks").fetchall()}
project_ids_projects = {r[0] for r in conn.execute("SELECT project_id FROM projects").fetchall()}
print(f"project_ids in tasks: {project_ids_tasks}")
print(f"project_ids in projects: {project_ids_projects}")
print(f"All task project_ids exist in projects: {project_ids_tasks.issubset(project_ids_projects)}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/20 [00:00<?, ?it/s]

Generating reviews:   0%|          | 0/10 [00:00<?, ?it/s]

project_ids in tasks: {1, 2, 3, 4, 5, 6, 8, 9, 10}
project_ids in projects: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
All task project_ids exist in projects: True


## 6. ColumnAssociation 显式关联

当隐式 FK 无法推断时，通过 `ColumnAssociation` 显式声明跨表关联：

```yaml
associations:
  - column: member_no
    ref_table: members
    ref_column: member_no
```

详见 09-config-deep-dive.ipynb

## 7. 多表批量填充

使用 `fill_from_config` 从 YAML/JSON 配置批量填充多张表，自动按 FK 依赖排序：

In [7]:
from sqlseed import GeneratorConfig, ProviderType, TableConfig, fill_from_config
from sqlseed.config.loader import save_config

config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType("mimesis"),
    tables=[
        TableConfig(name="organizations", count=3),
        TableConfig(name="members", count=10),
        TableConfig(name="projects", count=5),
        TableConfig(name="tasks", count=30),
    ],
)

config_path = str(Path("batch_config.yaml"))
save_config(config, config_path)

results = fill_from_config(config_path, clear_before=True)
for r in results:
    print(f"{r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/30 [00:00<?, ?it/s]

organizations: 3 rows in 0.056s
members: 10 rows in 0.088s
projects: 5 rows in 0.029s
tasks: 30 rows in 0.040s


## 8. clear_before 与 FK 约束

`clear_before=True` 会先 DELETE 再 INSERT。注意：如果子表引用了父表的数据，需要先清子表再清父表。

sqlseed 的 `fill_from_config` 会自动按拓扑排序处理，但单独使用 `fill()` 时需要手动注意顺序。

## 9. BLOB 列处理

BLOB 类型的列会生成随机二进制数据：

In [8]:

# file_name 匹配 *_name 模式 -> name 生成器(生成人名)
# 这里用 columns 覆盖为更合理的文件名格式
rows = preview(str(db_path), table="attachments", count=2,
               columns={"file_name": {"type": "pattern", "regex": "[a-z]{8}[.](pdf|png|docx)"}})
for row in rows:
    print(f"file_name={row['file_name']}, uploaded_at={row['uploaded_at']}")
print("\nfile_data (BLOB, nullable) 和 file_size (DEFAULT 0) 被策略链跳过")
print("BLOB 列在 nullable 策略中默认跳过, 有 DEFAULT 的列使用默认值")

file_name=djknrqdb.docx, uploaded_at=2019-01-03 18:09:44.650555
file_name=svihpyfk.png, uploaded_at=2021-06-15 18:08:02.745692

file_data (BLOB, nullable) 和 file_size (DEFAULT 0) 被策略链跳过
BLOB 列在 nullable 策略中默认跳过, 有 DEFAULT 的列使用默认值


## 📋 DatabaseAdapter 完整 API

DatabaseAdapter Protocol 定义了所有数据库操作接口：

In [9]:
with sqlseed.connect(str(db_path)) as orch:
    print('=== get_table_names() ===')
    tables = orch.get_table_names()
    print(f'  Tables: {tables}')

    print('\n=== get_column_info(organizations) ===')
    col_info = orch.get_column_info('organizations')
    for col in col_info:
        print(f'  {col.name}: type={col.type}, pk={col.is_primary_key}')

    print('\n=== get_foreign_keys(tasks) ===')
    fks = orch.get_foreign_keys('tasks')
    for fk in fks:
        print(f'  {fk}')

    print('\n=== get_row_count() ===')
    for table in tables:
        count = orch.get_row_count(table)
        print(f'  {table}: {count} rows')

=== get_table_names() ===
  Tables: ['organizations', 'members', 'sqlite_sequence', 'projects', 'tasks', 'reviews', 'tags', 'task_tags', 'attachments']

=== get_column_info(organizations) ===
  org_code: type=VARCHAR(16), pk=True
  name: type=VARCHAR(64), pk=False
  parent_code: type=VARCHAR(16), pk=False
  description: type=TEXT, pk=False
  is_active: type=INTEGER, pk=False
  member_count: type=INTEGER, pk=False
  created_at: type=TEXT, pk=False

=== get_foreign_keys(tasks) ===
  ForeignKeyInfo(column='assignee_id', ref_table='members', ref_column='member_id')
  ForeignKeyInfo(column='project_id', ref_table='projects', ref_column='project_id')

=== get_row_count() ===
  organizations: 3 rows
  members: 10 rows
  sqlite_sequence: 5 rows
  projects: 5 rows
  tasks: 30 rows
  reviews: 10 rows
  tags: 5 rows
  task_tags: 0 rows
  attachments: 0 rows


## ⚡ PragmaOptimizer 三级优化参数

PragmaOptimizer 根据预期行数自动选择优化级别：

| 级别 | 触发条件 | 关键 PRAGMA |
|---|---|---|
| Light | < 1,000 行 | journal_mode=WAL, synchronous=NORMAL |
| Moderate | 1,000-10,000 行 | + cache_size=-32000, temp_store=MEMORY |
| Aggressive | > 10,000 行 | + mmap_size=512MB, locking_mode=EXCLUSIVE |

执行完毕后自动恢复原始 PRAGMA 设置。

In [10]:

print("PragmaOptimizer 三级优化参数:")
print("  Light 阈值: < 1,000 行")
print("  Moderate 阈值: 1,000 - 10,000 行")
print("  Aggressive 阈值: > 10,000 行")

with sqlseed.connect(str(db_path), optimize_pragma=True) as orch:
    result = orch.fill_table("organizations", count=5, clear_before=True)
    print(f"\n填充 {result.count} 行 (Light 级别)")

PragmaOptimizer 三级优化参数:
  Light 阈值: < 1,000 行
  Moderate 阈值: 1,000 - 10,000 行
  Aggressive 阈值: > 10,000 行


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]


填充 5 行 (Light 级别)


## 10. 总结

| 特性 | 说明 |
|------|------|
| 双适配器 | sqlite-utils（默认）/ raw SQLite（降级） |
| Pragma 优化 | 三级策略，自动选择 |
| 显式 FK | CREATE TABLE 中声明 |
| 隐式 FK | 列名与父表主键同名自动推断 |
| SharedPool | 跨表引用一致性 |
| ColumnAssociation | 显式声明关联 |
| fill_from_config | 自动拓扑排序，批量填充 |
| BLOB | 随机二进制数据 |

**下一步**: [05-dag-and-constraints.ipynb](05-dag-and-constraints.ipynb) — DAG 拓扑排序与约束求解

In [11]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
